# V1 — Aspect-Decomposed User Representations

Instead of one mean vector per user, represent each user as a matrix of aspect vectors:
one vector per concept axis (cuisine, ingredient, diet, etc.).

This fixes mean collapse by keeping taste dimensions separate —
a user who likes both Thai and Italian gets distinct cuisine, ingredient, and diet vectors
instead of one averaged blob.

Scoring: a candidate recipe's score is a weighted sum of per-aspect similarities.

In [1]:
import pandas as pd
import numpy as np
import ast
from pathlib import Path
from collections import Counter, defaultdict

DATA_DIR = Path("../data")

In [2]:
recipes = pd.read_csv(DATA_DIR / "RAW_recipes.csv")
interactions = pd.read_csv(DATA_DIR / "RAW_interactions.csv")
recipe_embeddings = np.load(DATA_DIR / "recipe_embeddings_v0.npy")

def parse_list_str(s):
    try:
        return ast.literal_eval(s)
    except (ValueError, SyntaxError):
        return []

recipes['tags_list'] = recipes['tags'].apply(parse_list_str)
recipes['ingredients_list'] = recipes['ingredients'].apply(parse_list_str)
recipe_id_to_idx = dict(zip(recipes['id'], range(len(recipes))))

print(f"Recipes: {len(recipes):,}")
print(f"Interactions: {len(interactions):,}")
print(f"Embeddings: {recipe_embeddings.shape}")

Recipes: 231,637
Interactions: 1,132,367
Embeddings: (231637, 384)


## 1. Define concept taxonomy

Map Food.com's 552 tags into a small set of aspect axes.
Each axis captures a different dimension of taste.

Tags that don't map to a taste-relevant axis (like `time-to-make`, `equipment`,
`number-of-servings`) are dropped — they describe the recipe's logistics, not its flavor.

In [3]:
# Concept axes and which tags belong to each.
# A recipe can have tags in multiple axes.
# A tag can only belong to one axis (no double-counting).

CONCEPT_TAXONOMY = {
    'cuisine': {
        'north-american', 'american', 'southern-united-states', 'southwestern-united-states',
        'northeastern-united-states', 'midwestern', 'pacific-northwest', 'californian',
        'cajun', 'creole', 'tex-mex', 'hawaiian', 'canadian', 'ontario',
        'british-columbian', 'quebec', 'european', 'italian', 'french', 'spanish',
        'greek', 'german', 'english', 'irish', 'scottish', 'welsh', 'scandinavian',
        'swedish', 'norwegian', 'danish', 'finnish', 'icelandic', 'dutch', 'belgian',
        'swiss', 'austrian', 'hungarian', 'polish', 'czech', 'russian', 'portuguese',
        'asian', 'chinese', 'japanese', 'thai', 'indian', 'korean', 'vietnamese',
        'indonesian', 'malaysian', 'filipino', 'cambodian', 'laotian', 'szechuan',
        'cantonese', 'hunan', 'beijing', 'pakistani', 'nepalese', 'mongolian',
        'mexican', 'caribbean', 'cuban', 'puerto-rican', 'central-american',
        'south-american', 'brazilian', 'argentine', 'peruvian', 'colombian',
        'chilean', 'ecuadorean', 'venezuelan', 'costa-rican', 'guatemalan',
        'honduran', 'baja', 'oaxacan',
        'middle-eastern', 'turkish', 'lebanese', 'iranian-persian', 'iraqi',
        'saudi-arabian', 'palestinian', 'egyptian', 'moroccan', 'ethiopian',
        'african', 'south-african', 'nigerian', 'libyan', 'sudanese', 'angolan',
        'congolese', 'somalian', 'namibian', 'georgian',
        'south-west-pacific', 'australian', 'new-zealand', 'polynesian',
        'micro-melanesia',
        'jewish-ashkenazi', 'jewish-sephardi', 'amish-mennonite',
        'native-american', 'pennsylvania-dutch', 'soul',
    },
    'dish_type': {
        'main-dish', 'side-dishes', 'desserts', 'appetizers', 'snacks',
        'breakfast', 'brunch', 'lunch', 'salads', 'soups-stews', 'stews',
        'breads', 'sandwiches', 'beverages', 'cocktails', 'smoothies',
        'condiments-etc', 'sauces', 'savory-sauces', 'sweet-sauces',
        'dips', 'spreads', 'salsas', 'marinades-and-rubs',
        'casseroles', 'chili', 'curries', 'gumbo', 'chowders',
        'pasta', 'pizza', 'one-dish-meal', 'finger-food',
        'cookies-and-brownies', 'cakes', 'pies-and-tarts', 'pies',
        'tarts', 'cheesecake', 'cupcakes', 'brownies', 'candy', 'fudge',
        'frozen-desserts', 'puddings-and-mousses', 'gelatin',
        'pancakes-and-waffles', 'muffins', 'scones', 'quick-breads',
        'rolls-biscuits', 'coffee-cakes', 'granola-and-porridge',
        'cobblers-and-crisps', 'ice-cream', 'shakes',
        'salad-dressings', 'stuffings-dressings', 'garnishes',
        'jams-and-preserves', 'jellies', 'chutneys',
        'clear-soups', 'bisques-cream-soups', 'stocks',
        'stir-fry', 'lasagna', 'spaghetti', 'manicotti',
        'ravioli-tortellini', 'pasta-shells', 'penne', 'elbow-macaroni',
    },
    'protein': {
        'meat', 'poultry', 'chicken', 'chicken-breasts', 'chicken-thighs-legs',
        'whole-chicken', 'chicken-livers',
        'beef', 'ground-beef', 'roast-beef', 'beef-ribs', 'steaks', 'steak',
        'beef-organ-meats', 'beef-liver',
        'pork', 'pork-chops', 'pork-loins', 'pork-ribs', 'pork-sausage',
        'ham', 'bacon', 'beef-sausage',
        'turkey', 'turkey-breasts', 'whole-turkey', 'duck', 'duck-breasts',
        'whole-duck', 'quail', 'pheasant', 'goose',
        'lamb-sheep', 'veal', 'wild-game', 'deer', 'elk', 'moose',
        'rabbit', 'bear',
        'seafood', 'fish', 'saltwater-fish', 'freshwater-fish',
        'shrimp', 'crab', 'lobster', 'scallops', 'clams', 'mussels',
        'oysters', 'squid', 'octopus', 'crawfish',
        'salmon', 'tuna', 'catfish', 'tilapia', 'halibut', 'cod',
        'bass', 'trout', 'perch', 'sole-and-flounder', 'mahi-mahi',
        'orange-roughy', 'whitefish',
        'shellfish',
        'soy-tofu', 'tempeh', 'eggs', 'eggs-dairy', 'cheese',
    },
    'produce': {
        'vegetables', 'fruit', 'beans', 'lentils', 'chick-peas-garbanzos',
        'black-beans', 'green-yellow-beans',
        'potatoes', 'yams-sweet-potatoes', 'corn', 'rice',
        'long-grain-rice', 'white-rice', 'brown-rice', 'short-grain-rice',
        'medium-grain-rice', 'grains', 'pasta-rice-and-grains',
        'tomatoes', 'onions', 'mushrooms', 'peppers', 'broccoli',
        'spinach', 'cauliflower', 'asparagus', 'carrots', 'squash',
        'greens', 'lettuces', 'chard', 'collard-greens', 'bok-choys',
        'eggplant', 'artichoke', 'zucchini', 'avocado', 'pumpkin',
        'berries', 'strawberries', 'blueberries', 'raspberries', 'cherries',
        'tropical-fruit', 'pineapple', 'mango', 'papaya', 'coconut',
        'citrus', 'lemon', 'lime', 'oranges',
        'apples', 'pears', 'peaches', 'plums', 'grapes', 'melons',
        'bananas', 'kiwifruit', 'pitted-fruit',
        'nuts', 'peanut-butter',
        'chocolate',
    },
    'dietary': {
        'vegetarian', 'vegan', 'gluten-free', 'kosher', 'diabetic',
        'healthy', 'healthy-2', 'low-sodium', 'low-carb', 'very-low-carbs',
        'low-cholesterol', 'low-calorie', 'low-protein', 'low-saturated-fat',
        'low-fat', 'low-in-something', 'high-protein', 'high-calcium',
        'high-fiber', 'high-in-something', 'free-of-something',
        'egg-free', 'lactose', 'dairy-free', 'nut-free', 'no-shell-fish',
        'inexpensive',
    },
    'taste_mood': {
        'taste-mood', 'comfort-food', 'spicy', 'savory', 'sweet',
        'served-hot', 'served-cold',
    },
}

ASPECT_NAMES = list(CONCEPT_TAXONOMY.keys())

# Invert: tag → aspect
tag_to_aspect = {}
for aspect, tags in CONCEPT_TAXONOMY.items():
    for tag in tags:
        tag_to_aspect[tag] = aspect

# Verify no tag is in multiple aspects
total_tags = sum(len(tags) for tags in CONCEPT_TAXONOMY.values())
assert total_tags == len(tag_to_aspect), "Some tags appear in multiple aspects!"

print(f"Aspects: {ASPECT_NAMES}")
print(f"Total classified tags: {len(tag_to_aspect)}")
print(f"Tags per aspect:")
for name in ASPECT_NAMES:
    print(f"  {name}: {len(CONCEPT_TAXONOMY[name])}")

Aspects: ['cuisine', 'dish_type', 'protein', 'produce', 'dietary', 'taste_mood']
Total classified tags: 353
Tags per aspect:
  cuisine: 109
  dish_type: 73
  protein: 73
  produce: 64
  dietary: 27
  taste_mood: 7


## 2. Tag recipes by aspect

For each recipe, extract which tags it has in each aspect axis.
Most recipes will have tags in several aspects (a butter chicken has cuisine=indian, protein=chicken, dietary=none, etc.).

In [4]:
def get_recipe_aspects(tags_list):
    """Map a recipe's tags to aspect axes. Returns {aspect: [tags]}."""
    aspects = defaultdict(list)
    for tag in tags_list:
        if tag in tag_to_aspect:
            aspects[tag_to_aspect[tag]].append(tag)
    return dict(aspects)

recipes['aspects'] = recipes['tags_list'].apply(get_recipe_aspects)

# Coverage: how many recipes have at least one tag in each aspect?
print("Aspect coverage across recipes:")
for name in ASPECT_NAMES:
    count = sum(1 for aspects in recipes['aspects'] if name in aspects)
    print(f"  {name}: {count:,} ({count/len(recipes)*100:.1f}%)")

Aspect coverage across recipes:
  cuisine: 91,102 (39.3%)
  dish_type: 219,286 (94.7%)
  protein: 94,992 (41.0%)
  produce: 113,583 (49.0%)
  dietary: 133,426 (57.6%)
  taste_mood: 60,634 (26.2%)


In [5]:
# Spot-check a few recipes
for i in [0, 100, 5000]:
    row = recipes.iloc[i]
    print(f"\n{row['name']}")
    for aspect, tags in row['aspects'].items():
        print(f"  {aspect}: {tags}")


arriba   baked winter squash mexican style
  cuisine: ['north-american', 'mexican']
  dish_type: ['side-dishes']
  produce: ['vegetables', 'squash']
  dietary: ['vegetarian']

tide me over   indian chaat  simple veggie salad
  dietary: ['low-protein', 'healthy', 'low-fat', 'vegetarian', 'low-sodium', 'low-cholesterol', 'low-saturated-fat', 'low-calorie', 'low-carb', 'inexpensive', 'healthy-2', 'low-in-something']
  dish_type: ['lunch', 'salads', 'snacks']
  produce: ['vegetables', 'tomatoes']
  cuisine: ['asian', 'indian']
  taste_mood: ['spicy', 'taste-mood', 'savory', 'served-cold']

amish oatmeal cookies   slice and bake in a roll
  cuisine: ['north-american', 'american', 'amish-mennonite', 'northeastern-united-states']
  dish_type: ['desserts', 'cookies-and-brownies']


## 3. Temporal split + positive interactions

Same split as V0 for comparable evaluation.

In [6]:
interactions['date'] = pd.to_datetime(interactions['date'])
interactions = interactions.sort_values('date')

split_idx = int(len(interactions) * 0.8)
train_interactions = interactions.iloc[:split_idx]
test_interactions = interactions.iloc[split_idx:]

train_positive = train_interactions[train_interactions['rating'] >= 4].copy()
test_positive = test_interactions[test_interactions['rating'] >= 4].copy()

common_users = set(train_positive['user_id']) & set(test_positive['user_id'])
print(f"Train positive: {len(train_positive):,}")
print(f"Test positive: {len(test_positive):,}")
print(f"Common users: {len(common_users):,}")

Train positive: 822,501
Test positive: 181,223
Common users: 10,000


## 4. Build per-aspect user embeddings

For each user and each aspect axis:
1. Collect all recipes the user liked that have tags in this aspect
2. Weighted-mean their full recipe embeddings (weighted by rating)
3. Normalize

The result is a matrix per user: (n_aspects × 384).

Note: we're still using the full recipe embedding as the vector, not a per-aspect sub-embedding.
The aspect decomposition comes from *which recipes contribute* to each axis,
not from slicing the embedding dimensions. This means the cuisine vector for a user
who likes Indian food will point toward Indian recipes in the full embedding space.

In [7]:
RATING_WEIGHTS = {4: 1.0, 5: 2.0}

user_train_data = train_positive.groupby('user_id').agg(
    recipe_ids=('recipe_id', list),
    ratings=('rating', list),
)

def build_aspect_embeddings(recipe_ids, ratings):
    """Build one embedding per aspect from a user's liked recipes.
    Returns dict {aspect: normalized_embedding} and set of seen recipe ids."""
    # Group recipes by which aspects they participate in
    aspect_indices = defaultdict(list)
    aspect_weights = defaultdict(list)
    
    for rid, rating in zip(recipe_ids, ratings):
        if rid not in recipe_id_to_idx:
            continue
        idx = recipe_id_to_idx[rid]
        w = RATING_WEIGHTS.get(rating, 1.0)
        recipe_aspects = recipes.iloc[idx]['aspects']
        
        for aspect in recipe_aspects:
            aspect_indices[aspect].append(idx)
            aspect_weights[aspect].append(w)
    
    aspect_embs = {}
    for aspect in ASPECT_NAMES:
        if aspect in aspect_indices:
            indices = aspect_indices[aspect]
            weights = np.array(aspect_weights[aspect])
            emb = (recipe_embeddings[indices] * weights[:, None]).sum(axis=0) / weights.sum()
            emb = emb / np.linalg.norm(emb)
            aspect_embs[aspect] = emb
    
    return aspect_embs

In [8]:
user_aspect_embs = {}
user_seen_recipes = {}

for user_id in common_users:
    if user_id not in user_train_data.index:
        continue
    row = user_train_data.loc[user_id]
    aspect_embs = build_aspect_embeddings(row['recipe_ids'], row['ratings'])
    if aspect_embs:
        user_aspect_embs[user_id] = aspect_embs
        user_seen_recipes[user_id] = set(row['recipe_ids'])

print(f"Built aspect embeddings for {len(user_aspect_embs):,} users")

# How many aspects does a typical user have?
aspect_counts = [len(embs) for embs in user_aspect_embs.values()]
print(f"Aspects per user: mean={np.mean(aspect_counts):.1f}, min={np.min(aspect_counts)}, max={np.max(aspect_counts)}")

Built aspect embeddings for 9,995 users
Aspects per user: mean=5.2, min=1, max=6


## 5. Aspect-decomposed retrieval

For each candidate recipe, compute per-aspect similarity to the user,
then combine. Only aspects where both user and recipe have signal contribute.

Scoring: `score(user, recipe) = mean of per-aspect cosine similarities`
(averaged over aspects that both user and recipe participate in).

In [9]:
K = 10
RETRIEVE_POOL = 50
recipe_emb_matrix = recipe_embeddings.astype(np.float32)
recipe_ids_arr = recipes['id'].values

# Precompute per-recipe aspect membership (which aspects does each recipe have?)
recipe_aspect_mask = np.zeros((len(recipes), len(ASPECT_NAMES)), dtype=bool)
for i, aspects in enumerate(recipes['aspects']):
    for j, name in enumerate(ASPECT_NAMES):
        if name in aspects:
            recipe_aspect_mask[i, j] = True

print(f"Recipe-aspect mask shape: {recipe_aspect_mask.shape}")
print(f"Mean aspects per recipe: {recipe_aspect_mask.sum(axis=1).mean():.1f}")

Recipe-aspect mask shape: (231637, 6)
Mean aspects per recipe: 3.1


In [10]:
user_ids = list(user_aspect_embs.keys())
aspect_recommendations = {}

for progress, uid in enumerate(user_ids):
    user_embs = user_aspect_embs[uid]
    
    # Compute per-aspect similarity: for each aspect the user has,
    # dot product against all recipes
    aspect_sims = np.zeros(len(recipes), dtype=np.float32)
    aspect_counts = np.zeros(len(recipes), dtype=np.float32)
    
    for j, name in enumerate(ASPECT_NAMES):
        if name not in user_embs:
            continue
        # Similarity of this user's aspect vector to all recipes
        sims = recipe_emb_matrix @ user_embs[name].astype(np.float32)
        # Only count for recipes that also have this aspect
        mask = recipe_aspect_mask[:, j]
        aspect_sims[mask] += sims[mask]
        aspect_counts[mask] += 1.0
    
    # Average over contributing aspects
    valid = aspect_counts > 0
    scores = np.zeros(len(recipes), dtype=np.float32)
    scores[valid] = aspect_sims[valid] / aspect_counts[valid]
    
    top_pool_idx = np.argpartition(-scores, RETRIEVE_POOL)[:RETRIEVE_POOL]
    sorted_idx = top_pool_idx[np.argsort(-scores[top_pool_idx])]
    
    seen = user_seen_recipes.get(uid, set())
    recs = [recipe_ids_arr[j] for j in sorted_idx if recipe_ids_arr[j] not in seen][:K]
    aspect_recommendations[uid] = recs
    
    if (progress + 1) % 2000 == 0:
        print(f"  {progress + 1}/{len(user_ids)} users...")

print(f"Generated top-{K} recommendations for {len(aspect_recommendations):,} users")

  2000/9995 users...
  4000/9995 users...
  6000/9995 users...
  8000/9995 users...
Generated top-10 recommendations for 9,995 users


## 6. Evaluation — compare all methods

In [11]:
def recall_at_k(recommended, relevant):
    if not relevant:
        return 0.0
    return len(set(recommended) & set(relevant)) / len(relevant)

def hit_rate(recommended, relevant):
    return 1.0 if set(recommended) & set(relevant) else 0.0

def ndcg_at_k(recommended, relevant):
    dcg = 0.0
    for i, item in enumerate(recommended):
        if item in relevant:
            dcg += 1.0 / np.log2(i + 2)
    ideal_hits = min(len(relevant), len(recommended))
    idcg = sum(1.0 / np.log2(i + 2) for i in range(ideal_hits))
    return dcg / idcg if idcg > 0 else 0.0

test_user_likes = test_positive.groupby('user_id')['recipe_id'].apply(set).to_dict()

def evaluate(recs_dict, label):
    recalls, hits, ndcgs = [], [], []
    for uid in recs_dict:
        relevant = test_user_likes.get(uid, set())
        if not relevant:
            continue
        recs = recs_dict[uid]
        recalls.append(recall_at_k(recs, relevant))
        hits.append(hit_rate(recs, relevant))
        ndcgs.append(ndcg_at_k(recs, relevant))
    return {
        'method': label,
        'users': len(recalls),
        'recall@10': np.mean(recalls),
        'hitrate@10': np.mean(hits),
        'ndcg@10': np.mean(ndcgs),
    }

In [12]:
results = evaluate(aspect_recommendations, 'Aspect-decomposed')

print(f"Evaluated on {results['users']:,} users")
print()
print(f"{'Method':<25} {'Recall@10':>10} {'HitRate@10':>12} {'nDCG@10':>10}")
print(f"{'-'*57}")
print(f"{'V0 mean embedding':<25} {'0.0009':>10} {'0.0023':>12} {'0.0007':>10}")
print(f"{'V0 max-sim':<25} {'0.0012':>10} {'0.0027':>12} {'0.0008':>10}")
print(f"{results['method']:<25} {results['recall@10']:>10.4f} {results['hitrate@10']:>12.4f} {results['ndcg@10']:>10.4f}")

Evaluated on 9,995 users

Method                     Recall@10   HitRate@10    nDCG@10
---------------------------------------------------------
V0 mean embedding             0.0009       0.0023     0.0007
V0 max-sim                    0.0012       0.0027     0.0008
Aspect-decomposed             0.0011       0.0027     0.0007


## 7. Sanity check — inspect a user's aspect profile

In [13]:
sample_uid = user_ids[42]
sample_aspects = user_aspect_embs[sample_uid]
sample_seen = user_seen_recipes[sample_uid]

print(f"User {sample_uid}")
print(f"Liked {len(sample_seen)} recipes in train")
print(f"Has embeddings for aspects: {list(sample_aspects.keys())}")

# For each aspect, show the top-3 most similar recipes
for aspect in ASPECT_NAMES:
    if aspect not in sample_aspects:
        continue
    sims = recipe_emb_matrix @ sample_aspects[aspect].astype(np.float32)
    mask = recipe_aspect_mask[:, ASPECT_NAMES.index(aspect)]
    sims[~mask] = -1
    # Exclude seen
    for rid in sample_seen:
        if rid in recipe_id_to_idx:
            sims[recipe_id_to_idx[rid]] = -1
    top3 = np.argsort(-sims)[:3]
    print(f"\n  {aspect}:")
    for idx in top3:
        print(f"    {sims[idx]:.3f}  {recipes.iloc[idx]['name']}")

print(f"\nFinal top-{K} recommendations:")
for rid in aspect_recommendations[sample_uid]:
    idx = recipe_id_to_idx.get(rid)
    if idx is not None:
        print(f"  - {recipes.iloc[idx]['name']}")

test_likes = test_user_likes.get(sample_uid, set())
if test_likes:
    print(f"\nActually liked in test ({len(test_likes)} recipes):")
    for rid in list(test_likes)[:5]:
        if rid in recipe_id_to_idx:
            print(f"  - {recipes.iloc[recipe_id_to_idx[rid]]['name']}")

User 163986
Liked 64 recipes in train
Has embeddings for aspects: ['cuisine', 'dish_type', 'protein', 'produce', 'dietary', 'taste_mood']

  cuisine:
    0.850  caramelized onion and white bean flatbread
    0.843  savory croissant breakfast pudding
    0.838  whole wheat pasta with greens  beans and pancetta

  dish_type:
    0.852  cheese and squeeze   cheddar and beef   biscuit balls
    0.846  caramelized onion and white bean flatbread
    0.838  sausage and roasted peppers pasta bake

  protein:
    0.880  wonderful beef and noodle casserole
    0.875  sausage and roasted peppers pasta bake
    0.866  baked meatballs and pasta

  produce:
    0.847  rice and mushroom delight
    0.841  cheddar and veggie bread pudding
    0.840  caramelized onion and white bean flatbread

  dietary:
    0.838  corn  cheddar  and sun dried tomato muffins
    0.835  caramelized onion and white bean flatbread
    0.834  beefy biscuit casserole

  taste_mood:
    0.838  whole wheat pasta with greens  